# Mapping Notebook

This notebook supports the geographic presentation portion of the BUSA 695 capstone project. It combines the cleaned project data with a simple modeling dataset, trains a random forest model for prediction, attaches prediction errors back to property records, adds latitude and longitude from zip codes, and builds an interactive housing map.

**Datasets loaded:** `realtor_master.csv` and `cleaned_simple.csv`

**Main steps:**
- Load the cleaned datasets used for mapping and prediction.
- Train a random forest model on the simplified feature set.
- Add predicted prices and pricing errors to the mapping dataset.
- Save the trained model file.
- Geocode zip codes to latitude and longitude.
- Create an interactive map for presentation.

**Outputs created:** `rf_model.pkl` and `housing_map.html`

**Capstone connection:** This notebook turns the project’s modeling work into a map-based deliverable that helps explain geographic pricing patterns in the final capstone presentation.
            

In [1]:
import os
import csv
import pandas as pd 


c:\Users\A1990\anaconda3\envs\dev\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Load Prepared Datasets

This section brings together the cleaned master data for map details and the simplified dataset used to train the random forest model.
            

In [2]:
# Load the cleaned datasets used to connect model predictions with map-ready property records.
# Load datasets
raw_df = pd.read_csv("realtor_master.csv")
model_df = pd.read_csv("cleaned_simple.csv")

# IMPORTANT: make sure both datasets align (same rows after cleaning)
# If you filtered rows earlier, we may need to re-run cleaning pipeline on raw_df

# For now, assume same order (we will adjust if needed)
map_df = raw_df.copy()

C:\Users\A1990\AppData\Local\Temp\ipykernel_8804\181147635.py:3: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv("realtor_master.csv")


In [3]:
print(raw_df.shape)
print(model_df)

(2224841, 11)
            price  bed  bath  acre_lot  house_size  status_for_sale  \
0        105000.0  3.0   2.0      0.12       920.0             True   
1         80000.0  4.0   2.0      0.08      1527.0             True   
2         67000.0  2.0   1.0      0.15       748.0             True   
3        145000.0  4.0   2.0      0.10      1800.0             True   
4         65000.0  6.0   2.0      0.05      1760.0             True   
...           ...  ...   ...       ...         ...              ...   
2224836  359900.0  4.0   2.0      0.33      3600.0            False   
2224837  350000.0  3.0   2.0      0.10      1616.0            False   
2224838  440000.0  6.0   3.0      0.50      3200.0            False   
2224839  179900.0  2.0   1.0      0.09       933.0            False   
2224840  580000.0  5.0   3.0      0.31      3615.0            False   

         status_ready_to_build  status_sold  
0                        False        False  
1                        False        Fal

## Train a Prediction Model for Mapping

These steps prepare the features, train the random forest model, and generate price predictions that can be linked back to the map records.
            

In [4]:
# Train a random forest model so predicted prices and errors can be displayed geographically.
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Prepare data
X = model_df.drop("price", axis=1)
y = model_df["price"]

# Split (same as before)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=25
)

# Train model
model = RandomForestRegressor(
    n_estimators=500,  # keep it reasonable so it runs fast
    random_state=25,
    n_jobs=-1
)

model.fit(X_train, y_train)

,n_estimators,500
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [5]:
map_df["predicted_price"] = model.predict(X)

In [6]:
map_df["error"] = map_df["price"] - map_df["predicted_price"]

In [7]:
import joblib
joblib.dump(model, "rf_model.pkl")

['rf_model.pkl']

In [8]:
map_df[["price", "predicted_price", "error"]].head()

,price,predicted_price,error
0,105000.0,161239.937000,-56239.937000
1,80000.0,118842.110952,-38842.110952
2,67000.0,96183.911400,-29183.911400
3,145000.0,183646.852088,-38646.852088
4,65000.0,655988.021149,-590988.021149


## Add Geographic Coordinates

This section converts zip codes into latitude and longitude values so each sampled property can be placed on a map.
            

In [9]:
# Create and save the interactive HTML map used in the capstone presentation.
map_df_sample = map_df.sample(1000, random_state= 25)

In [10]:
import pgeocode
import pandas as pd

nomi = pgeocode.Nominatim('us')

# Clean ZIP codes
map_df_sample["zip_code"] = map_df_sample["zip_code"].fillna(0)
map_df_sample["zip_code"] = map_df_sample["zip_code"].astype(float).astype(int)

# Get coords
def get_coords(zip_code):
    if zip_code == 0:
        return pd.Series([None, None])
    
    result = nomi.query_postal_code(str(zip_code))
    
    #Fix
    if pd.isna(result.latitude) or pd.isna(result.longitude):
        return pd.Series([None, None])
    
    return pd.Series([result.latitude, result.longitude])

map_df_sample[["lat", "lon"]] = map_df_sample["zip_code"].apply(get_coords)

In [11]:
map_df_sample[["zip_code", "lat", "lon"]].head(10)

,zip_code,lat,lon
1706903,33511,27.9056,-82.2881
515930,33559,28.1801,-82.4169
872616,60402,41.8347,-87.7914
1569427,21801,38.3824,-75.6336
27944,4238,NaN,NaN
797474,53402,42.7726,-87.7960
1377551,97330,44.5904,-123.2722
269585,24504,37.3610,-79.0544
509634,33469,26.9831,-80.1080
2126477,94116,37.7441,-122.4863


## Build the Interactive Housing Map

The final map uses the prediction results and geocoded locations to create a visual project output for exploration and presentation.
            

In [12]:
import folium

m = folium.Map(
    location=[37.8, -96],
    zoom_start=4,
    tiles="CartoDB positron"  
)

for _, row in map_df_sample.iterrows():
    if pd.isna(row["lat"]) or pd.isna(row["lon"]):
        continue

    abs_error = abs(row["error"])

    if abs_error <= 100000:
        marker_color = "green"
    elif abs_error <= 250000:
        marker_color = "orange"
    else:
        marker_color = "red"

    tooltip_text = (
        f"{row['city']}, {row['state']}<br>"
        f"Absolute Error: ${abs_error:,.0f}"
    )

    popup_html = f"""
    <b>City:</b> {row['city']}<br>
    <b>State:</b> {row['state']}<br>
    <b>Actual Price:</b> ${row['price']:,.0f}<br>
    <b>Predicted Price:</b> ${row['predicted_price']:,.0f}<br>
    <b>Prediction Error:</b> ${row['error']:,.0f}<br>
    <b>Absolute Error:</b> ${abs_error:,.0f}
    """

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=7,
        color=marker_color,
        fill=True,
        fill_color=marker_color,
        fill_opacity=0.65,
        weight=1,
        tooltip=folium.Tooltip(tooltip_text),
        popup=folium.Popup(popup_html, max_width=300),
    ).add_to(m)

legend_html = """
<div style="
position: fixed;
bottom: 50px; left: 50px; width: 260px; height: 130px;
background-color: white; z-index:9999; font-size:14px;
border:2px solid grey; padding:10px;
border-radius:5px;
">
<b>Prediction Error Legend</b><br>
<span style="color:green; font-size:16px;">?</span> Low error (? $100,000)<br>
<span style="color:orange; font-size:16px;">?</span> Moderate error ($100,001?$250,000)<br>
<span style="color:red; font-size:16px;">?</span> High error (> $250,000)
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m.save("housing_map.html")